In [10]:
from elasticsearch import Elasticsearch
from langchain_ollama import OllamaEmbeddings
es = Elasticsearch("http://localhost:9200")
embed_model=OllamaEmbeddings(base_url="http://localhost:11434", model="bge-m3:latest")
es

d:\auto_vectordb\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Elasticsearch(['http://localhost:9200'])>

In [6]:
indices_dict = es.indices.get_alias(index="ai*")
index_names = list(indices_dict.keys())
index_names

['ai_paper']

# Embed Search

In [ ]:
def embed_search(index_name:str, query_text: str = "", size: int = 10, min_score: float = 0.5):
    """
    Elasticsearch에서 텍스트 또는 임베딩을 기반으로 문서를 검색합니다.
    
    텍스트 검색 (query_text)과 벡터 검색 (query_embedding) 중 하나 또는 둘 다를 사용하여 검색할 수 있습니다.
    둘 다 제공되면 부스트 값이 적용된 Bool 쿼리 (RRF)를 구성합니다.
    
    Args:
        query_text (str, optional): 일반 텍스트 검색어. Defaults to None.
        query_embedding (list, optional): 벡터 검색을 위한 1024차원 임베딩 리스트. Defaults to None.
        size (int, optional): 반환할 최대 문서 수. Defaults to 10.
        min_score (float, optional): 최소 점수 임계값. Defaults to 0.5.
        
    Returns:
        list: 검색 결과 문서 (hit['_source'] + score) 리스트.
    """
    search_body = {
        "size": size,
        "min_score": min_score,
        "query": {
            "bool": {
                "should": [],
                "minimum_should_match": 1 # 'should' 절 중 최소 하나는 일치해야 함
            }
        },
        # Elasticsearch 8.x 이상에서 kNN 검색을 위한 kNN 섹션 추가 (Elasticsearch 버전에 따라 달라질 수 있음)
        "knn": []
    }
    
    # 1. 일반 텍스트 검색 쿼리 (Query Text)
    if query_text:
        # "page_content" 필드에서 텍스트를 검색하는 match 쿼리 추가
        search_body["query"]["bool"]["should"].append({
            "match": {
                "page_content": {
                    "query": query_text,
                    "boost": 1.0 # 텍스트 검색 부스트 값
                }
            }
        })
        print(f"Text search enabled for: {query_text}")
    
    # 2. 벡터 검색 쿼리 (Query Embedding)
    query_embedding = embed_model.embed_query(query_text)_
    if query_embedding:
        if len(query_embedding) != 1024:
            print(f"Embedding must be 1024 dimensions, got {len(query_embedding)}")
            return []
        
        # kNN 섹션에 dense_vector 검색 추가
        # 참고: Elasticsearch 8.x 버전에서는 search API의 'knn' 파라미터를 사용하거나
        # 7.x 버전에서는 'script_score' 쿼리를 사용할 수 있습니다.
        # 여기서는 8.x의 'knn' 파라미터를 사용하는 표준 방식을 따릅니다.
        search_body["knn"].append({
            "field": "embeddings",
            "query_vector": query_embedding,
            "k": size, # k: 이웃 수
            "num_candidates": max(size * 10, 50), # 검색할 후보 수 (성능/정확도 트레이드오프)
            "boost": 0.8 # 벡터 검색 부스트 값 (텍스트 검색보다 약간 낮게 설정)
        })
        
        # kNN을 사용할 경우, 최소 점수 대신 필터링을 사용하여 관련 없는 문서를 제거할 수 있습니다.
        # 이 예시에서는 min_score를 유지합니다.
        
        print("Vector search enabled.")
        
    try:
        # 3. 검색 실행
        # Elasticsearch 8.x에서는 kNN과 쿼리를 조합할 수 있습니다.
        # 'knn' 파라미터가 비어 있지 않으면 'search_body'에서 'knn'을 제거하고 별도의 'knn' 인수로 전달해야 합니다.
        # 하지만 8.x 클라이언트의 search 메서드가 body에 knn을 허용하는 경우가 많으므로 body에 포함합니다.
        res = es.search(
            index=index_name, 
            body=search_body
        )
        
        # 4. 결과 파싱 및 반환
        hits = res['hits']['hits']
        
        # 결과에 점수 (Relevance Score)를 포함하여 반환합니다.
        documents = [{'_score': hit['_score'], **hit['_source']} for hit in hits]
        
        print(f"Found {len(documents)} documents.")
        
        return documents
        
    except Exception as e:
        print(f"Error searching documents: {e}")
        return []

In [14]:
index_name = "ai_paper"
query_text = "auto recursive regression"
semantic_docs = embed_search(index_name=index_name, query_text=query_text, size= 10, min_score=0.1)
semantic_docs

Text search enabled for: auto recursive regression
Found 10 documents.


[{'_score': 4.354237,
  'id': '4f0f7183-54eb-4fe0-9996-d2d819ae0e09',
  'page_content': "This page explains From Local to Global_ A Graph RAG Approach to Query-Focused Summarization_2404.16130v2 that belongs to Paper categories.\n- Metropolitansky, D. and Larson, J. (2025). Towards effective extraction and evaluation of factual claims.\n- Microsoft (2023). The impact of large language models on scientific discovery: a preliminary study using gpt-4.\n- Mooney, R. J. and Bunescu, R. (2005). Mining knowledge from text using information extraction. SIGKDD Explor. Newsl., 7(1):3–10.\n- NebulaGraph (2024). Nebulagraph launches industry-first graph rag: Retrieval-augmented generation with llm based on knowledge graphs. https://www . nebula-graph . io/posts/graph-RAG .\n- Neo4J (2024). Get started with graphrag: Neo4j's ecosystem tools. https://neo4j . com/developerblog/graphrag-ecosystem-tools/ .\n- Newman, M. E. (2006). Modularity and community structure in networks. Proceedings of the natio

# BM25

In [12]:
import spacy
import numpy as np
from rank_bm25 import BM25Okapi
from tqdm import tqdm

# 영어 모델 로드 (가벼운 sm 모델 또는 정밀한 trf 모델 선택 가능)
nlp = spacy.load("D:\\models\\en_core_web_sm", disable=["ner", "parser"])

d:\auto_vectordb\.venv\Lib\site-packages\spacy\util.py:969: UserWarning: [W095] Model 'en_core_web_sm' (3.7.1) was trained with spaCy v3.7.2 and may not be 100% compatible with the current version (3.8.11). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [19]:
# 3. spaCy 전처리 함수
def spacy_tokenizer(text):
    # 문서 객체 생성
    doc = nlp(text)
    # 1. 불용어(Stopwords) 제거
    # 2. 구두점(Punctuation) 제거 
    # 3. 표제어 추출(Lemmatization) 및 소문자화
    return [token.lemma_.lower() for token in doc 
            if not token.is_stop and not token.is_punct and not token.is_space]

def sigmoid(x, scaling_factor=10):
    """
    BM25 점수를 0~1 사이로 변환합니다.
    scaling_factor가 클수록 점수 차이가 완만하게 반영됩니다.
    """
    return 1 / (1 + np.exp(-np.array(x) / scaling_factor))

def bm25_search(corpus: list, query: str, top_k: int = 3):
    # 1. 전처리 및 토큰화
    print("Tokenizing documents...")
    tokenized_corpus = []
    # n_process=-1을 위해 if __name__ == "__main__": 블록 내 실행 권장
    for doc in tqdm(nlp.pipe(corpus, batch_size=2000, n_process=-1), total=len(corpus)):
        tokens = [token.lemma_.lower() for token in doc if not token.is_stop and not token.is_punct]
        tokenized_corpus.append(tokens)

    # 2. BM25 인덱싱
    bm25 = BM25Okapi(tokenized_corpus)

    # 3. 쿼리 전처리 및 점수 계산
    tokenized_query = spacy_tokenizer(query)
    doc_scores = bm25.get_scores(tokenized_query)

    # 4. Sigmoid 정규화 적용
    # 보통 BM25 상위 점수가 10~20 내외이므로 scaling_factor를 10 정도로 잡으면 적절합니다.
    normalized_scores = sigmoid(doc_scores, scaling_factor=10)

    # 5. 상위 K개 인덱스 추출
    top_n_indices = np.argsort(doc_scores)[::-1][:top_k]

    # 6. 결과 구성 (문서 내용, 원본 점수, 정규화 점수)
    results = []
    for idx in top_n_indices:
        results.append({
            "page_content": corpus[idx],
            "score": np.round(doc_scores[idx], 4),
            "similarity": np.round(normalized_scores[idx], 4)
        })

    return results

In [20]:
corpus = [c["page_content"] for c in semantic_docs]
corpus

["This page explains From Local to Global_ A Graph RAG Approach to Query-Focused Summarization_2404.16130v2 that belongs to Paper categories.\n- Metropolitansky, D. and Larson, J. (2025). Towards effective extraction and evaluation of factual claims.\n- Microsoft (2023). The impact of large language models on scientific discovery: a preliminary study using gpt-4.\n- Mooney, R. J. and Bunescu, R. (2005). Mining knowledge from text using information extraction. SIGKDD Explor. Newsl., 7(1):3–10.\n- NebulaGraph (2024). Nebulagraph launches industry-first graph rag: Retrieval-augmented generation with llm based on knowledge graphs. https://www . nebula-graph . io/posts/graph-RAG .\n- Neo4J (2024). Get started with graphrag: Neo4j's ecosystem tools. https://neo4j . com/developerblog/graphrag-ecosystem-tools/ .\n- Newman, M. E. (2006). Modularity and community structure in networks. Proceedings of the national academy of sciences, 103(23):8577–8582.\n- Ni, J., Shi, M., Stammbach, D., Sachan, 

In [21]:
bm25_docs = bm25_search(corpus=corpus, query=query_text, top_k=5)
bm25_docs

Tokenizing documents...


100%|██████████| 10/10 [00:58<00:00,  5.89s/it]


[{'page_content': 'This page explains Learning Transferable Visual Models From Natural Language Supervision_2103.00020v1 that belongs to Paper categories.\nFigure 6. Zero-shot CLIP outperforms few-shot linear probes. Zero-shot CLIP matches the average performance of a 4-shot linear classifier trained on the same feature space and nearly matches the best results of a 16-shot linear classifier across publicly available models. For both BiT-M and SimCLRv2, the best performing model is highlighted. Light gray lines are other models in the eval suite. The 20 datasets with at least 16 examples per class were used in this analysis.\n\n\n\nwe see that zero-shot CLIP is quite weak on several specialized, complex, or abstract tasks such as satellite image classification (EuroSAT and RESISC45), lymph node tumor detection (PatchCamelyon), counting objects in synthetic scenes (CLEVRCounts), self-driving related tasks such as German traffic sign recognition (GTSRB), recognizing distance to the neare

# RRF

In [23]:
bm25_rank = []
for d in bm25_docs:
    for s in semantic_docs:
        if s["page_content"] ==  d["page_content"]:
            bm25_rank.append(s["id"])
bm25_rank

['2b463ba9-3e44-46b7-ac42-14742aa216ef',
 'de618629-1d8d-4ad3-8009-3484ae228b10',
 '9bc41059-6e57-4056-ac66-a477be558837',
 '4f0f7183-54eb-4fe0-9996-d2d819ae0e09',
 '1fd79e3f-3ee5-44b9-bff5-485f64e72be6']

In [24]:
embed_rank = [d["id"]for d in semantic_docs]
embed_rank

['4f0f7183-54eb-4fe0-9996-d2d819ae0e09',
 '1fd79e3f-3ee5-44b9-bff5-485f64e72be6',
 '2b463ba9-3e44-46b7-ac42-14742aa216ef',
 'c8c9f29d-8d94-4f9f-99f7-6baedf80c27a',
 'fb8d52f2-df17-472e-8b99-67570f6e28cc',
 '8540751b-2dec-4150-8703-47fd7d0abe45',
 'de618629-1d8d-4ad3-8009-3484ae228b10',
 '40747a5e-d562-49a1-9c4f-fc5d16f30332',
 'ef99e9c6-cd74-4afe-b040-705288d849cf',
 '9bc41059-6e57-4056-ac66-a477be558837']

In [25]:
def reciprocal_rank_fusion(search_results, k=60):
    """
    search_results: 리스트의 리스트 (예: [[doc1, doc2], [doc2, doc3]])
    k: 평활화 상수 (기본값 60)
    """
    rrf_scores = {}

    for rank_list in search_results:
        for rank, doc_id in enumerate(rank_list):
            # rank는 0부터 시작하므로 순위 계산을 위해 +1
            actual_rank = rank + 1
            
            if doc_id not in rrf_scores:
                rrf_scores[doc_id] = 0
            
            # RRF 공식 적용
            rrf_scores[doc_id] += 1 / (k + actual_rank)

    # 점수가 높은 순서대로 정렬
    sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_results

In [26]:
sorted_results = reciprocal_rank_fusion([embed_rank, bm25_rank])
sorted_results

[('2b463ba9-3e44-46b7-ac42-14742aa216ef', 0.032266458495966696),
 ('4f0f7183-54eb-4fe0-9996-d2d819ae0e09', 0.032018442622950824),
 ('1fd79e3f-3ee5-44b9-bff5-485f64e72be6', 0.0315136476426799),
 ('de618629-1d8d-4ad3-8009-3484ae228b10', 0.031054405392392875),
 ('9bc41059-6e57-4056-ac66-a477be558837', 0.030158730158730156),
 ('c8c9f29d-8d94-4f9f-99f7-6baedf80c27a', 0.015625),
 ('fb8d52f2-df17-472e-8b99-67570f6e28cc', 0.015384615384615385),
 ('8540751b-2dec-4150-8703-47fd7d0abe45', 0.015151515151515152),
 ('40747a5e-d562-49a1-9c4f-fc5d16f30332', 0.014705882352941176),
 ('ef99e9c6-cd74-4afe-b040-705288d849cf', 0.014492753623188406)]